# PneumoScan — MobileNetV2 Transfer Learning

This notebook applies **MobileNetV2** as a feature extractor for chest X-ray pneumonia classification.  
MobileNetV2 is built around **inverted residual blocks with depthwise separable convolutions**, dramatically reducing computation and model size while maintaining competitive accuracy.  
Its lightweight architecture makes it especially relevant for **edge deployment** scenarios—mobile phones, embedded devices, or resource-constrained clinical settings where a full-scale model cannot run.

**Classes:** `NORMAL`, `BACTERIA`, `VIRUS` &nbsp;|&nbsp; **Input size:** 224 × 224 px

In [ ]:
# === PneumoScan Setup ===
import os, sys, subprocess

# Detect environment and configure paths
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO_DIR = '/content/PneumoScan'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/muhammadrakib2299/PneumoScan.git', REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    
    SRC_DIR = os.path.join(REPO_DIR, 'src')
    IN_COLAB = True
    print(f"Running on Google Colab | src: {SRC_DIR}")
except ImportError:
    SRC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'src')
    IN_COLAB = False
    print(f"Running locally | src: {SRC_DIR}")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# === Imports ===
import config
import data_loader
import models
import train
import evaluate
import utils

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Configure paths for Colab (dataset on Google Drive)
if IN_COLAB:
    config.setup_colab()

config.ensure_dirs()

print(f"Project root: {config.BASE_DIR}")
print(f"Dataset dir:  {config.RAW_DATA_DIR}")
print(f"Models dir:   {config.MODELS_DIR}")

## 1 — Data Loading

We load the training and validation splits and compute balanced class weights to address the uneven distribution across NORMAL, BACTERIA, and VIRUS samples.

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Load train / validation datasets
train_ds, val_ds = data_loader.load_train_val_split()

# Compute class weights from training labels
train_labels = np.concatenate([y.numpy() for _, y in train_ds], axis=0)
if train_labels.ndim > 1:          # one-hot → integer labels
    train_labels = np.argmax(train_labels, axis=1)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(config.CLASS_NAMES)),
    y=train_labels,
)
class_weights = dict(enumerate(class_weights_array))

print("Class weights:")
for idx, name in enumerate(config.CLASS_NAMES):
    print(f"  {name:>10s}: {class_weights[idx]:.4f}")

## 2 — Model Architecture

### Why MobileNetV2?

MobileNetV2 (Sandler et al., 2018) builds on two core ideas:

**1. Depthwise Separable Convolutions**  
A standard convolution with kernel $K \times K$ over $C_{in}$ input channels producing $C_{out}$ output channels costs $K^2 \cdot C_{in} \cdot C_{out}$ multiply-adds per spatial position.  
Depthwise separable convolutions factorise this into:
- A **depthwise** convolution ($K^2 \cdot C_{in}$) that filters each channel independently, and  
- A **pointwise** $1 \times 1$ convolution ($C_{in} \cdot C_{out}$) that combines channels.

This reduces computation by a factor of roughly $\frac{1}{C_{out}} + \frac{1}{K^2}$, which for a $3 \times 3$ kernel is approximately **8–9 times fewer** operations.

**2. Inverted Residual Blocks**  
Unlike ResNet residuals that compress then expand, MobileNetV2 *expands* channels with a $1 \times 1$ convolution, applies the depthwise convolution in the high-dimensional space, and then *projects* back to a narrow bottleneck. Residual shortcuts connect the narrow bottleneck layers.

| Property | Value |
|----------|-------|
| Parameters | ~3.4 M |
| Multiply-Adds | ~300 M |
| Top-1 ImageNet | ~72% |

### Edge deployment relevance

In many clinical environments—rural health posts, mobile screening vans, or smartphone-based triage apps—server-grade GPUs are unavailable.  
MobileNetV2 can be converted to **TensorFlow Lite** or **ONNX** and run real-time inference on a phone CPU, making it the natural candidate for PneumoScan’s edge deployment path.

In [ ]:
# Build MobileNetV2 with frozen ImageNet backbone
model = models.build_mobilenetv2(freeze=True)
model.summary()

## 3 — Training

We train only the classification head on top of the frozen MobileNetV2 features. All callbacks (early stopping, learning-rate reduction, checkpointing) are managed by `train.train_model()`.

In [ ]:
history = train.train_model(
    model_name="mobilenetv2",
    train_ds=train_ds,
    val_ds=val_ds,
    class_weights=class_weights,
)

print("Training complete.")

## 4 — Evaluation

We evaluate the trained model on the held-out test set using `evaluate.evaluate_model()`, which generates the classification report, confusion matrix, and per-class diagnostic plots.

In [ ]:
# Load test dataset
test_ds = data_loader.load_test_data()

# Full evaluation (metrics + plots)
results = evaluate.evaluate_model(
    model=model,
    test_ds=test_ds,
    model_name="mobilenetv2",
)

print("Test evaluation finished.")

## Summary

MobileNetV2 offers the most lightweight backbone in our transfer-learning comparison.  
Key take-aways:

* **Extreme efficiency** — At ~3.4 M parameters and ~300 M multiply-adds, MobileNetV2 trains and infers significantly faster than EfficientNet-B0 or DenseNet-121, making it ideal for rapid prototyping and resource-limited environments.
* **Edge-ready** — The architecture was designed from the ground up for mobile and embedded deployment. A TFLite-converted MobileNetV2 can run real-time chest X-ray classification on a smartphone, enabling point-of-care screening in low-resource settings.
* **Depthwise separable convolutions** — Factorising standard convolutions into depthwise and pointwise steps cuts computation by nearly an order of magnitude with only a modest accuracy trade-off.
* **Trade-off awareness** — If test metrics show a meaningful gap compared to EfficientNet-B0 or DenseNet-121, consider partial fine-tuning (unfreezing the last few blocks) or using MobileNetV2 as a lightweight ensemble member alongside a heavier model.

Compare these results with the EfficientNet-B0 and DenseNet-121 notebooks to select the optimal backbone for PneumoScan’s deployment target.